# Querying notebook

Here we create a notebook separate from the ingestion process to call on the database already created, to see how the query process works with the persistent search index setup.

## Set up query 

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

Check how many documents are in the index:

In [2]:
sqlite_index.count()

340

We can run the cell above multiple times while the other notebook is ingesting data.  We will see the number of documents increase as long as the ingestion notebook is running.

Let's try a search:

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?']

_Note:  I ran the ingestion command cell in the other notebook using a different record number request each time (e.g., [:60]--Alexey's default, [:100], [:150], [:40]) to see the number of records increase in this notebook.  Since the faq.db is persistent, each time I ran the ingestion command, the same documents were added because I started each call at index 0.  Something to keep in mind when setting up the ingestion process for an end-to-end app._

## RAG with sqlitesearch

We use the ```RAGBase``` class from ```rag_helper.py``` with this sqlitesearch index.

Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

In [4]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client
)

This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, so we just connect to the database file.

Try it:



In [5]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


This answer should be the same as we saw with ```minsearch```.  In this case, however, the data comes from a persistent index.  This modular design split works as follows:
* ```ingest.py``` handles data loading and indexing
* ```rag_helper.py``` handles the RAG pipeline
* The notebooks connect them

This works because ```sqlitesearch``` follows the same API as ```minsearch``` - both have a search method that takes a query, ```boost_dict```, ```filter_dict```, and ```num_results```. If the API were different, we'd need to subclass ```RAGBase``` and override the ```search``` method to adapt to the new backend.

## Comparing the two approaches

With minsearch (single process):

```Startup: fetch data -> parse -> index -> ready```

```Every restart: repeat all steps```

With sqlitesearch (two processes):

```Ingestion (runs once): fetch data -> parse -> write to faq.db```

```Query (runs every time): open faq.db -> search -> ready```

For our FAQ dataset, both produce good results. The difference matters more at scale with diverse document lengths.



## Choosing an approach

Pick the right tool for your data:

* ```minsearch```: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
* ```sqlitesearch```: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.

Use minsearch when you can load and index the data on startup without noticeable delay. Switch to a persistent backend when ingestion takes too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different backend:

* Elasticsearch
* OpenSearch
* Qdrant (vector database)
* Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

## Cleaning up

When we're done, we should close the database connection.  This can be done with a command...


In [7]:
sqlite_index.close()

...or simply when the notebook is shutdown (python closes the connection).